# Aplicando os métodos de regressão

Carregamos os dados que já foram pré-processados e submetidos à Redução de Dimensionalidade (PCA). É fundamental separar as variáveis identificadoras (TEAM_ID e SEASON_ID) da matriz de features para garantir que os modelos não utilizem essas informações de forma indevida durante o treinamento (evitando vazamento de dados). Por fim, isolamos a nossa variável alvo: PLAYOFF_WINS.

In [ ]:
import pandas as pd

# carregando os dados (Features reduzidas e Alvo)
X_train_full = pd.read_csv('../CSVs/X_train_pca.csv')
X_test_full = pd.read_csv('../CSVs/X_test_pca.csv')
y_train_df = pd.read_csv('../CSVs/y_train_final.csv')
y_test_df = pd.read_csv('../CSVs/y_test_final.csv')

# separando os identificadores das features (O modelo não pode ver os IDs)
colunas_id = ['TEAM_ID', 'SEASON_ID']

X_train = X_train_full.drop(columns=colunas_id)
X_test = X_test_full.drop(columns=colunas_id)

# convertendo o alvo para Pandas Series (formato exigido pelo Scikit-Learn)
y_train = y_train_df['PLAYOFF_WINS']
y_test = y_test_df['PLAYOFF_WINS']

# verificação 
print("--- Dados carregados e prontos para o Torneio de Modelos ---")
print(f"Treino: Matriz X = {X_train.shape} | Vetor y = {y_train.shape}")
print(f"Teste:  Matriz X = {X_test.shape}   | Vetor y = {y_test.shape}")
print(f"\nVariáveis preditoras ativas: {X_train.columns.tolist()}")

--- Dados carregados e prontos para o Torneio de Modelos ---
Treino: Matriz X = (802, 5) | Vetor y = (802,)
Teste:  Matriz X = (30, 5)   | Vetor y = (30,)

Variáveis preditoras ativas: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5']


Para garantir uma comparação justa e direta entre todas as técnicas de regressão que testaremos, criamos uma função auxiliar unificada. Ela calcula as principais métricas de erro (MAE, MSE e RMSE) e a métrica de ajuste (R²). Essa função também nos ajudará a estruturar um quadro comparativo ao final de todas as execuções.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# criando a Função Padrão de Avaliação (Para usar em todos os modelos)
def avaliar_modelo(nome_modelo, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    print(f"--- Resultados: {nome_modelo} ---")
    print(f"MAE  (Erro médio absoluto):      {mae:.4f}")
    print(f"MSE  (Penaliza erros grandes):   {mse:.4f}")
    print(f"RMSE (Mesma unidade de y):       {rmse:.4f}")
    print(f"R²   (Ganho sobre a média):      {r2:.4f}")
    
    # retorna um dicionário para podermos montar uma tabela comparativa no final
    return {'Modelo': nome_modelo, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}


Iniciamos o nosso torneio de modelos estabelecendo um baseline (linha de base) com a Regressão Linear. Este algoritmo tentará encontrar a melhor relação linear entre os nossos componentes principais (PCs) e o número de vitórias nos playoffs. É um modelo mais simples, rápido e que nos dará uma boa noção inicial do comportamento dos dados.

In [ ]:
# ==========================================
# treinando e avaliando a Regressão Linear
# ==========================================

lr_pca = LinearRegression()
lr_pca.fit(X_train, y_train)

y_pred_lr = lr_pca.predict(X_test)

# chamando a nossa função de avaliação
metricas_lr = avaliar_modelo("Regressão Linear (PCA-5)", y_test, y_pred_lr)


--- Resultados: Regressão Linear (PCA-5) ---
MAE  (Erro médio absoluto):      2.6231
MSE  (Penaliza erros grandes):   10.9812
RMSE (Mesma unidade de y):       3.3138
R²   (Ganho sobre a média):      0.3759


Como isolamos os identificadores no início, criamos este dicionário auxiliar para traduzir os IDs numéricos da NBA de volta para os nomes reais das franquias. Isso facilitará a interpretação qualitativa dos resultados nas nossas tabelas de previsão.

In [ ]:
# dicionário padrão de IDs da NBA para os nomes das franquias
# pra melhorar visualização posteriormente
nba_teams = {
    1610612737: 'Atlanta Hawks', 1610612738: 'Boston Celtics', 1610612739: 'Cleveland Cavaliers',
    1610612740: 'New Orleans Pelicans', 1610612741: 'Chicago Bulls', 1610612742: 'Dallas Mavericks',
    1610612743: 'Denver Nuggets', 1610612744: 'Golden State Warriors', 1610612745: 'Houston Rockets',
    1610612746: 'LA Clippers', 1610612747: 'Los Angeles Lakers', 1610612748: 'Miami Heat',
    1610612749: 'Milwaukee Bucks', 1610612750: 'Minnesota Timberwolves', 1610612751: 'Brooklyn Nets',
    1610612752: 'New York Knicks', 1610612753: 'Orlando Magic', 1610612754: 'Indiana Pacers',
    1610612755: 'Philadelphia 76ers', 1610612756: 'Phoenix Suns', 1610612757: 'Portland Trail Blazers',
    1610612758: 'Sacramento Kings', 1610612759: 'San Antonio Spurs', 1610612760: 'Oklahoma City Thunder',
    1610612761: 'Toronto Raptors', 1610612762: 'Utah Jazz', 1610612763: 'Memphis Grizzlies',
    1610612764: 'Washington Wizards', 1610612765: 'Detroit Pistons', 1610612766: 'Charlotte Hornets'
}

Abaixo, unimos as previsões geradas pelo modelo linear com o número real de vitórias na temporada de teste. Ordenar esses dados nos ajuda a identificar rapidamente quais times o modelo conseguiu prever com maior precisão e onde ele superestimou ou subestimou o desempenho.

In [ ]:

# visualizando as previsões
print("\n--- Times (Ordenados por Vitórias Reais) ---")
resultados_lr = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_lr['VITORIAS_REAIS'] = y_test.values
resultados_lr['PREVISAO_LR'] = y_pred_lr.round(1)

# aplicando o mapeamento no DataFrame de resultados
resultados_lr['TEAM_NAME'] = resultados_lr['TEAM_ID'].map(nba_teams)

# reorganizando as colunas para o nome aparecer logo depois do ID e facilitar a leitura
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_LR']
resultados_lr = resultados_lr[colunas_ordem]

display(resultados_lr.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Times (Ordenados por Vitórias Reais) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_LR
1,1610612738,Boston Celtics,2023-24,16,8.7
6,1610612742,Dallas Mavericks,2023-24,13,3.8
17,1610612750,Minnesota Timberwolves,2023-24,9,6.9
11,1610612754,Indiana Pacers,2023-24,8,3.9
19,1610612752,New York Knicks,2023-24,7,5.0
7,1610612743,Denver Nuggets,2023-24,7,6.1
20,1610612760,Oklahoma City Thunder,2023-24,6,6.5
5,1610612739,Cleveland Cavaliers,2023-24,5,4.6
21,1610612753,Orlando Magic,2023-24,3,4.6
16,1610612749,Milwaukee Bucks,2023-24,2,4.4


Nosso segundo modelo introduz não-linearidade. A Árvore de Decisão particiona o espaço das variáveis do PCA para fazer suas estimativas. Para evitar que o algoritmo simplesmente "decore" os dados de treino (overfitting) e perca a capacidade de generalização, limitamos o crescimento da árvore definindo o parâmetro de profundidade máxima (max_depth=5).

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# inicializando a Árvore de Decisão
# Usamos max_depth para evitar que a árvore cresça infinitamente e "decore" o treino (overfitting)
# O random_state garante que o resultado seja o mesmo toda vez que rodar
dt_pca = DecisionTreeRegressor(max_depth=5, random_state=42)

# treinando o Modelo
dt_pca.fit(X_train, y_train)

# fazendo as Previsões
y_pred_dt = dt_pca.predict(X_test)

# avaliando usando a nossa função
metricas_dt = avaliar_modelo("Decision Tree (PCA-5)", y_test, y_pred_dt)

--- Resultados: Decision Tree (PCA-5) ---
MAE  (Erro médio absoluto):      2.7360
MSE  (Penaliza erros grandes):   22.0040
RMSE (Mesma unidade de y):       4.6908
R²   (Ganho sobre a média):      -0.2505


Comparativo estruturado das previsões feitas pela Árvore de Decisão frente aos resultados reais da temporada.

In [ ]:

# visualizando as previsões
print("\n--- Times (Ordenados por Vitórias Reais) ---")
resultados_dt = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_dt['VITORIAS_REAIS'] = y_test.values
resultados_dt['PREVISAO_DT'] = y_pred_dt.round(1)

# aplicando o mapeamento no DataFrame de resultados
resultados_dt['TEAM_NAME'] = resultados_dt['TEAM_ID'].map(nba_teams)

# reorganizando as colunas para o nome aparecer logo depois do ID e facilitar a leitura
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_DT']
resultados_dt = resultados_dt[colunas_ordem]

display(resultados_dt.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Times (Ordenados por Vitórias Reais) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_DT
1,1610612738,Boston Celtics,2023-24,16,6.6
6,1610612742,Dallas Mavericks,2023-24,13,0.8
17,1610612750,Minnesota Timberwolves,2023-24,9,9.7
11,1610612754,Indiana Pacers,2023-24,8,4.5
19,1610612752,New York Knicks,2023-24,7,5.2
7,1610612743,Denver Nuggets,2023-24,7,10.5
20,1610612760,Oklahoma City Thunder,2023-24,6,10.5
5,1610612739,Cleveland Cavaliers,2023-24,5,5.2
21,1610612753,Orlando Magic,2023-24,3,3.5
16,1610612749,Milwaukee Bucks,2023-24,2,0.8


Para mitigar a alta variância característica de uma única Árvore de Decisão, evoluímos para um método ensemble: o Random Forest. Este modelo cria múltiplas árvores de decisão durante o treinamento (neste caso, 100 estimadores) e calcula a média das previsões de todas elas, entregando, geralmente, um resultado mais robusto e estável.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# inicializando o Random Forest
# n_estimators: Número de árvores na floresta (100 é um bom padrão)
# max_depth: Profundidade máxima de cada árvore
# random_state: Para garantir reprodutibilidade
rf_pca = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

# treinando o Modelo
rf_pca.fit(X_train, y_train)

# fazendo as Previsões
y_pred_rf = rf_pca.predict(X_test)

# avaliando usando a nossa função
metricas_rf = avaliar_modelo("Random Forest (PCA-5)", y_test, y_pred_rf)

--- Resultados: Random Forest (PCA-5) ---
MAE  (Erro médio absoluto):      2.3398
MSE  (Penaliza erros grandes):   12.6735
RMSE (Mesma unidade de y):       3.5600
R²   (Ganho sobre a média):      0.2797


Análise do desempenho preditivo do modelo Random Forest em relação às vitórias reais observadas nos playoffs.

In [ ]:
# visualizando as previsões
print("\n--- Times (Ordenados por Vitórias Reais) ---")
resultados_rf = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_rf['TEAM_NAME'] = resultados_rf['TEAM_ID'].map(nba_teams)
resultados_rf['VITORIAS_REAIS'] = y_test.values
resultados_rf['PREVISAO_RF'] = y_pred_rf.round(1)

# reorganizando as colunas
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_RF']
resultados_rf = resultados_rf[colunas_ordem]

display(resultados_rf.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Times (Ordenados por Vitórias Reais) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_RF
1,1610612738,Boston Celtics,2023-24,16,7.5
6,1610612742,Dallas Mavericks,2023-24,13,3.9
17,1610612750,Minnesota Timberwolves,2023-24,9,9.3
11,1610612754,Indiana Pacers,2023-24,8,5.4
19,1610612752,New York Knicks,2023-24,7,5.8
7,1610612743,Denver Nuggets,2023-24,7,9.1
20,1610612760,Oklahoma City Thunder,2023-24,6,10.0
5,1610612739,Cleveland Cavaliers,2023-24,5,4.9
21,1610612753,Orlando Magic,2023-24,3,2.7
16,1610612749,Milwaukee Bucks,2023-24,2,4.0
